In [ ]:
!pip install -q faiss-gpu psutil
import warnings
warnings.filterwarnings('ignore')
print('faiss-gpu + psutil ready.')


In [ ]:
import os
import json
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project"
PHASE3_DIR  = f"{PROJECT_DIR}/phase3_outputs"
PHASE4_DIR  = f"{PROJECT_DIR}/phase4_outputs"
os.makedirs(PHASE4_DIR, exist_ok=True)

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_records     = load_jsonl(f"{PHASE3_DIR}/train.jsonl")
test_open_records = load_jsonl(f"{PHASE3_DIR}/test_open.jsonl")
train_texts    = [r['text']    for r in train_records]
train_poets    = [r['poet']    for r in train_records]
train_poem_ids = [r['poem_id'] for r in train_records]
print(f"train: {len(train_records):,} | test_open: {len(test_open_records):,}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5EncoderModel, ByT5Tokenizer

MODEL_NAME = "google/byt5-base"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

class ByT5PACLEncoder(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-8)
        z      = F.normalize(pooled, dim=1)
        return z

tokenizer = ByT5Tokenizer.from_pretrained(MODEL_NAME)
model     = ByT5PACLEncoder().to(DEVICE)

ckpt_path = (
    f"{PHASE3_DIR}/checkpoints_byt5base/last_pacl_byt5baseEpoch9.pt"
)

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    state_dict = ckpt.get("model_state_dict", ckpt.get("model", None))
    if state_dict is None:
        raise KeyError(f"Cannot find model weights in checkpoint. Keys: {list(ckpt.keys())}")
    model.load_state_dict(state_dict)
    epoch_loaded = ckpt.get("epoch", 9)
    print(f"Loaded checkpoint: last_pacl_byt5baseEpoch9.pt  (epoch={epoch_loaded})")
else:
    raise FileNotFoundError(f"Checkpoint not found at: {ckpt_path}")

model.eval()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
print("Model ready.")


In [ ]:
import faiss
from sklearn.preprocessing import normalize as sk_normalize


def get_k(rec):
    try:
        return int(rec.get('k', rec.get('num_shots', -1)))
    except Exception:
        return -1

test_open_emb = np.load(f"{PHASE3_DIR}/test_open_embeddings_byt5base.npy")
k1_indices  = [i for i, r in enumerate(test_open_records) if get_k(r) == 1]
k1_records  = [test_open_records[i] for i in k1_indices]
k1_emb      = test_open_emb[k1_indices]
print(f"Gallery (Open k=1): {k1_emb.shape}")

gallery_norm = sk_normalize(k1_emb, norm='l2').astype(np.float32)
print(f"Gallery normalized: {gallery_norm.shape}  dtype={gallery_norm.dtype}")

D = gallery_norm.shape[1]
if DEVICE == 'cuda':
    res         = faiss.StandardGpuResources()
    flat_index  = faiss.IndexFlatIP(D)
    faiss_index = faiss.index_cpu_to_gpu(res, 0, flat_index)
else:
    faiss_index = faiss.IndexFlatIP(D)

faiss_index.add(gallery_norm)
print(f"FAISS index built: {faiss_index.ntotal:,} vectors  dim={D}")

if DEVICE == 'cuda':
    cpu_index = faiss.index_gpu_to_cpu(faiss_index)
else:
    cpu_index = faiss_index
FAISS_PATH = f"{PHASE4_DIR}/exp2_faiss_gallery.index"
faiss.write_index(cpu_index, FAISS_PATH)
faiss_sz = os.path.getsize(FAISS_PATH) / 1e6
print(f"FAISS index saved: {faiss_sz:.1f} MB")

ckpt_sz = os.path.getsize(ckpt_path) / 1e6 if os.path.exists(ckpt_path) else 0.0
print(f"ByT5 checkpoint size: {ckpt_sz:.1f} MB")

In [ ]:
import time
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 5),
    max_features=50000, sublinear_tf=True, dtype=np.float32
)
vectorizer.fit(train_texts)
print(f"TF-IDF fit: {len(vectorizer.vocabulary_):,} features")

k1_texts       = [r['text'] for r in k1_records]
gallery_sparse = vectorizer.transform(k1_texts)
from sklearn.preprocessing import normalize as sk_normalize
gallery_sparse_norm = sk_normalize(gallery_sparse, norm='l2')
print(f"Sparse gallery: {gallery_sparse_norm.shape}")

VECT_PATH  = f"{PHASE4_DIR}/exp2_tfidf_vectorizer.pkl"
SPARSE_PATH = f"{PHASE4_DIR}/exp2_tfidf_gallery.npz"
joblib.dump(vectorizer, VECT_PATH)
sp.save_npz(SPARSE_PATH, gallery_sparse)
vect_sz   = os.path.getsize(VECT_PATH)   / 1e6
sparse_sz = os.path.getsize(SPARSE_PATH) / 1e6
print(f"TF-IDF vectorizer: {vect_sz:.1f} MB")
print(f"TF-IDF sparse gallery: {sparse_sz:.1f} MB")


In [ ]:
import random

random.seed(42)
sample_size = 1000
sample_idx  = random.sample(range(len(k1_records)), sample_size)
sample_records = [k1_records[i] for i in sample_idx]
sample_texts   = [r['text'] for r in sample_records]

print(f"Sampled {len(sample_texts)} queries from Open-Set k=1")
print(f"Example: {sample_texts[0][:80]}...")


In [ ]:
import psutil

process = psutil.Process(os.getpid())
ram_mb  = process.memory_info().rss / 1e6

vram_mb = 0.0
if DEVICE == 'cuda':
    vram_mb = torch.cuda.max_memory_allocated() / 1e6

print("=" * 55)
print("STORAGE FOOTPRINT")
print("=" * 55)
print(f"  ByT5 checkpoint (disk)     : {ckpt_sz:>8.1f} MB")
print(f"  FAISS dense index (disk)   : {faiss_sz:>8.1f} MB")
print(f"  TF-IDF vectorizer (disk)   : {vect_sz:>8.1f} MB")
print(f"  TF-IDF sparse gallery (disk): {sparse_sz:>7.1f} MB")
print()
print("MEMORY FOOTPRINT (current session)")
print(f"  Process RAM                : {ram_mb:>8.1f} MB")
print(f"  Peak VRAM (GPU)            : {vram_mb:>8.1f} MB")

memory_log = {
    'ckpt_mb':   ckpt_sz,
    'faiss_mb':  faiss_sz,
    'vect_mb':   vect_sz,
    'sparse_mb': sparse_sz,
    'ram_mb':    ram_mb,
    'vram_peak_mb': vram_mb,
}


In [ ]:
times_sparse_enc    = []
times_sparse_search = []

print("Measuring TF-IDF latency (1,000 queries, batch_size=1) ...")
for text in sample_texts:
    t0    = time.perf_counter()
    s_vec = vectorizer.transform([text])
    s_vec_norm = sk_normalize(s_vec, norm='l2')
    t1    = time.perf_counter()
    times_sparse_enc.append((t1 - t0) * 1000)

    t0    = time.perf_counter()
    sims  = s_vec_norm @ gallery_sparse_norm.T
    if sp.issparse(sims):
        sims = sims.toarray()
    top500_sparse = np.argsort(-sims.ravel())[:500]
    t1    = time.perf_counter()
    times_sparse_search.append((t1 - t0) * 1000)

se_mean, se_std  = np.mean(times_sparse_enc),    np.std(times_sparse_enc)
ss_mean, ss_std  = np.mean(times_sparse_search), np.std(times_sparse_search)
st_mean          = se_mean + ss_mean
st_std           = np.sqrt(se_std**2 + ss_std**2)

print(f"  Enc  (sparse): {se_mean:.2f} +/- {se_std:.2f} ms")
print(f"  Search (sparse): {ss_mean:.2f} +/- {ss_std:.2f} ms")
print(f"  Total TF-IDF E2E: {st_mean:.2f} +/- {st_std:.2f} ms")


In [ ]:
times_dense_enc    = []
times_dense_search = []

if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()

print("Measuring ByT5 latency (1,000 queries, batch_size=1) ...")
with torch.no_grad():
    for text in sample_texts:
        t0  = time.perf_counter()
        enc = tokenizer(
            [text],
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=256
        ).to(DEVICE)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        d_vec = model(enc['input_ids'], enc['attention_mask'])  
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        d_vec_np = d_vec.cpu().numpy().astype(np.float32)       
        t1  = time.perf_counter()
        times_dense_enc.append((t1 - t0) * 1000)

        t0  = time.perf_counter()
        _, top500_dense = faiss_index.search(d_vec_np, 500)
        t1  = time.perf_counter()
        times_dense_search.append((t1 - t0) * 1000)

de_mean, de_std = np.mean(times_dense_enc),    np.std(times_dense_enc)
ds_mean, ds_std = np.mean(times_dense_search), np.std(times_dense_search)
dt_mean         = de_mean + ds_mean
dt_std          = np.sqrt(de_std**2 + ds_std**2)

vram_after_mb = torch.cuda.max_memory_allocated() / 1e6 if DEVICE == 'cuda' else 0.0

print(f"  Enc  (dense/ByT5): {de_mean:.2f} +/- {de_std:.2f} ms")
print(f"  Search (FAISS-GPU): {ds_mean:.2f} +/- {ds_std:.2f} ms")
print(f"  Total ByT5 E2E: {dt_mean:.2f} +/- {dt_std:.2f} ms")
print(f"  Peak VRAM after timing: {vram_after_mb:.1f} MB")


In [ ]:
times_rrf = []
BEST_MU   = 1.0

print("Measuring RRF fusion latency (1,000 queries, batch_size=1) ...")
with torch.no_grad():
    for text in sample_texts:
        s_vec      = vectorizer.transform([text])
        s_vec_norm = sk_normalize(s_vec, norm='l2')
        sims_s     = s_vec_norm @ gallery_sparse_norm.T
        if sp.issparse(sims_s):
            sims_s = sims_s.toarray()
        top500_s_idx = np.argsort(-sims_s.ravel())[:500]

        enc        = tokenizer(
            [text], return_tensors='pt', padding=True,
            truncation=True, max_length=256
        ).to(DEVICE)
        d_vec_np   = model(enc['input_ids'], enc['attention_mask']).cpu().numpy().astype(np.float32)
        _, top500_d_idx = faiss_index.search(d_vec_np, 500)
        top500_d_idx = top500_d_idx[0]

        t0 = time.perf_counter()
        from collections import defaultdict as _dd
        scores = _dd(float)
        for rank, gidx in enumerate(top500_d_idx.tolist(), 1):
            scores[gidx] += 1.0 / (60 + rank)
        for rank, gidx in enumerate(top500_s_idx.tolist(), 1):
            scores[gidx] += BEST_MU / (60 + rank)
        final_ranked = sorted(scores, key=lambda x: -scores[x])
        t1 = time.perf_counter()
        times_rrf.append((t1 - t0) * 1000)

fuse_mean, fuse_std = np.mean(times_rrf), np.std(times_rrf)
print(f"  RRF fusion overhead: {fuse_mean:.4f} +/- {fuse_std:.4f} ms")

tfidf_e2e_mean = se_mean + ss_mean
tfidf_e2e_std  = np.sqrt(se_std**2 + ss_std**2)
byt5_e2e_mean  = de_mean + ds_mean
byt5_e2e_std   = np.sqrt(de_std**2 + ds_std**2)
rrf_e2e_mean   = tfidf_e2e_mean + byt5_e2e_mean + fuse_mean
rrf_e2e_std    = np.sqrt(tfidf_e2e_std**2 + byt5_e2e_std**2 + fuse_std**2)

print()
print("=" * 65)
print("LATENCY SUMMARY (mean +/- std over 1,000 queries, batch_size=1)")
print("=" * 65)
print(f"  TF-IDF Enc:         {se_mean:>8.2f} +/- {se_std:.2f} ms")
print(f"  TF-IDF Search:      {ss_mean:>8.2f} +/- {ss_std:.2f} ms")
print(f"  ByT5 Enc:           {de_mean:>8.2f} +/- {de_std:.2f} ms")
print(f"  ByT5 Search(FAISS): {ds_mean:>8.2f} +/- {ds_std:.2f} ms")
print(f"  RRF Fusion:         {fuse_mean:>8.4f} +/- {fuse_std:.4f} ms")
print()
print(f"  TF-IDF Alone E2E:   {tfidf_e2e_mean:>8.2f} +/- {tfidf_e2e_std:.2f} ms")
print(f"  ByT5 Alone E2E:     {byt5_e2e_mean:>8.2f} +/- {byt5_e2e_std:.2f} ms")
print(f"  RRF (sequential) E2E: {rrf_e2e_mean:>6.2f} +/- {rrf_e2e_std:.2f} ms")


In [ ]:
import json as _json

def fmt(mean, std, unit='ms'):
    return f"{mean:.2f} $\\pm$ {std:.2f} {unit}"

mem_tfidf = f"{sparse_sz:.0f} MB (sparse matrix) + {vect_sz:.0f} MB (vectorizer)"
mem_byt5  = f"{ckpt_sz:.0f} MB (checkpoint) + {faiss_sz:.0f} MB (FAISS index)"
mem_rrf   = f"Sum of both: {ckpt_sz + faiss_sz + sparse_sz + vect_sz:.0f} MB total"

vram_str  = f"{vram_after_mb:.0f} MB VRAM (GPU)"
ram_str   = f"{ram_mb:.0f} MB RAM"

latex_table = r"""\begin{table}[t]
\centering
\caption{System Latency, Throughput, and Memory Footprint (Open-Set, batch\_size=1, $N$=1000 queries).}
\label{tab:latency}
\begin{tabular}{lccccc}
\toprule
\textbf{Pipeline} & \textbf{Index Size (Disk)} & \textbf{Memory (RAM/VRAM)} & \textbf{Feature Extraction} & \textbf{Search Latency} & \textbf{End-to-End} \\
\midrule
""" + \
f"TF-IDF Alone & {sparse_sz:.0f} MB & {ram_str} & {fmt(se_mean, se_std)} & {fmt(ss_mean, ss_std)} & {fmt(tfidf_e2e_mean, tfidf_e2e_std)} \\\\\n" + \
f"ByT5 Alone & {faiss_sz:.0f} MB & {vram_str} & {fmt(de_mean, de_std)} & {fmt(ds_mean, ds_std)} & {fmt(byt5_e2e_mean, byt5_e2e_std)} \\\\\n" + \
f"Multi-Granular (RRF) & {ckpt_sz+faiss_sz+sparse_sz+vect_sz:.0f} MB & {ram_str} / {vram_str} & {fmt(de_mean+se_mean, de_std+se_std)} & {fmt(ds_mean+ss_mean, ds_std+ss_std)} & {fmt(rrf_e2e_mean, rrf_e2e_std)} \\\\\n" + \
r"""\midrule
""" + \
f"\\quad + RRF Fusion Overhead & --- & --- & --- & --- & {fmt(fuse_mean, fuse_std)} \\\\\n" + \
r"""
\bottomrule
\end{tabular}
\end{table}"""

print("=== LaTeX Table ===")
print(latex_table)

out = {
    'experiment': 'Exp2_LatencyProfiling',
    'spec_source': 'EXP.txt (professor instructions)',
    'n_queries': 1000,
    'batch_size': 1,
    'gallery_size': faiss_index.ntotal,
    'best_mu_rrf': BEST_MU,
    'storage': memory_log,
    'vram_after_timing_mb': vram_after_mb,
    'latency_ms': {
        'sparse_enc':    {'mean': float(se_mean),         'std': float(se_std)},
        'sparse_search': {'mean': float(ss_mean),         'std': float(ss_std)},
        'dense_enc':     {'mean': float(de_mean),         'std': float(de_std)},
        'dense_search':  {'mean': float(ds_mean),         'std': float(ds_std)},
        'rrf_fusion':    {'mean': float(fuse_mean),       'std': float(fuse_std)},
        'tfidf_e2e':     {'mean': float(tfidf_e2e_mean),  'std': float(tfidf_e2e_std)},
        'byt5_e2e':      {'mean': float(byt5_e2e_mean),   'std': float(byt5_e2e_std)},
        'rrf_e2e':       {'mean': float(rrf_e2e_mean),    'std': float(rrf_e2e_std)},
    },
    'latex_table': latex_table,
}
out_path = f"{PHASE4_DIR}/exp2_latency_results.json"
with open(out_path, 'w', encoding='utf-8') as f:
    _json.dump(out, f, ensure_ascii=False, indent=2)
print(f"Saved: {out_path}")
